<a href="https://colab.research.google.com/github/Umang-Raval/DSSE_Assignment1/blob/main/Week3/Jinaai/%5BREAD_ONLY%5DDS4SE26Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0. Import requireed modules**

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
# ...

**1. Provide the path of java files and load their source code into a list**

In [ ]:
SOURCE_CODE_DIR = Path("/content/[UR_FOLDER_OF_JAVA_FILES]")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
files_list = []
# .rglob() searches the root folder and all sub-folders automatically
for file_path in SOURCE_CODE_DIR.rglob('*.java'):
    if file_path.is_file():
        with open(file_path, 'r', encoding='utf-8') as f:
            file_data.append(f.read())

**2. Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**3. Hardware optimization (Quantization)**

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

**4. Generate embeddings**

In [ ]:
embedding_model_name = "microsoft/codebert-base"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name,token=hf_token)
model = AutoModel.from_pretrained(embedding_model_name,token=hf_token,quantization_config=bnb_config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

**5. Construct the semantic similarity matrix**

In [ ]:
def embed_source_code(code_files):
    embeddings = []
    for code in code_files:

        # Max length 512 (for current model and available memory size)
        inputs = tokenizer(code, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)


        embeddings.append(outputs)
    return np.array(embeddings)

In [ ]:
embeddings = embed_source_code(files_list)
semantic_matrix = cosine_similarity(embeddings)

**6. Construct the structural similarity matrix**

In [ ]:
# scan the filtered .rsf dependency file from week 1 to construct a matrix where each entry is the number of packages each pair of files depend on.
# executing this cell produces 'struct_matrix_raw'

**7. Normalize the structural matrix**

In [ ]:
max_overlap = struct_matrix_raw.max()
struct_matrix = struct_matrix_raw / max_overlap if max_overlap > 0 else struct_matrix_raw
np.fill_diagonal(struct_matrix, 1.0)

**8. Combine the two matrices into one similarity matrix then apply complement.**

In [ ]:
# ALPHA dictates the weight of structural vs. semantic data.
# E.g., 0.4 means 40% Structural (RSF) and 60% Semantic (CodeBERT)
ALPHA = [HYPER_PARAMETER]
combined_similarity = (ALPHA * struct_matrix) + ((1 - ALPHA) * semantic_matrix)

# Invert similarity to get distance
distance_matrix = 1.0 - combined_similarity
np.fill_diagonal(distance_matrix, 0)

**9. Apply clustering**

In [ ]:
TARGET_NUM_CLUSTERS = [HYPER_PARAMETER] # how such parameter could be optimized?
clusterer = AgglomerativeClustering(n_clusters=TARGET_NUM_CLUSTERS, metric='precomputed', linkage='complete') # what is 'precomputed' & 'linkage'?
clusters = clusterer.fit_predict(distance_matrix)

**10. Visulizations**

In [ ]:
# According to the applied algorithm documentation, provide any visualizations for better understanding of how the clusters are formed.